# Notebook 04 · L2：状态驱动耦合 W(t)

**L2 = L1 + 耦合权重随状态变化 W(t) + 满足度判据 φ**。

这是本项目**自研创新点的第一层**（CSQ1）：文献里的耦合要么是 0/1（布尔），
要么是固定强度 ω；而现实是"**设备越差，这条耦合通道能供过来的越少**"。

⚠️ 写成"除以 1+ρ…"而不是"乘以 1+ρ…"：乘法的意思是退化节点反而拿到**更多**跨层供给
（机制变成稳定化），与"状态差 → 更容易崩"的假设相反。

    w_ij(t) = w0 / [1 + ρ·(1 − min(x_i, x_j))]

ρ=0 ⇒ W ≡ w0（常数耦合，退回 L1）；φ=0 ⇒ 没有服务质量停机 ⇒ ρ=0 且 φ=0 时退回 L0。

## 环境准备（每个 notebook 都要先跑这两格）

In [ ]:
"""环境准备：定位项目根目录、加 src 到搜索路径、设好画图参数。"""
import sys
from pathlib import Path

# 从当前工作目录往上找，直到看见 src/phase6_interdependent（这样在 notebooks/ 里也能跑）
root = Path.cwd()
while not (root / "src" / "phase6_interdependent").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

RESULTS = root / "results"
RESULTS.mkdir(exist_ok=True)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("项目根目录：", root)

In [ ]:
# 公共底座：一对 BA 双层网络 + 骨架类（后面所有 notebook 都从这一格开始）
from phase6_interdependent.core import InterdependentNetwork

N, m = 300, 3
G_A = nx.barabasi_albert_graph(N, m, seed=42)   # 层 A：比如电力网
G_B = nx.barabasi_albert_graph(N, m, seed=43)   # 层 B：比如通信网
nodes_A = sorted(G_A.nodes())
print(f"双层网络就绪：N={N}, 每层边数={G_A.number_of_edges()}")

## 第 1 步

In [ ]:
from phase6_interdependent.L2_state_driven_coupling import (
    StateDrivenConfig, coupling_weight, L2Rule
)

cfg = StateDrivenConfig(w0=1.0, rho=1.0)
for xi in [1.0, 0.8, 0.5, 0.2, 0.0]:
    print(f"x_i={xi:.1f} → w_ij = {coupling_weight(xi, 1.0, cfg):.3f}")
print("\nρ=0 时应当恒为 w0：")
cfg0 = StateDrivenConfig(w0=1.0, rho=0.0)
print([round(coupling_weight(x, 1.0, cfg0), 3) for x in [1.0, 0.5, 0.0]])

## 第 2 步

In [ ]:
# 退化性检验：ρ=0 且 φ=0 ⇒ 必须等于 L0
fractions = np.linspace(0.0, 0.3, 7)
rng = np.random.default_rng(42)
nodes_A = sorted(G_A.nodes())
cfg_deg = StateDrivenConfig(w0=1.0, rho=0.0, phi=0.0)

S_L0, S_L2 = [], []
for f in fractions:
    k = int(round(len(nodes_A) * f))
    a, b = [], []
    for _ in range(3):
        failed = ([] if k == 0 else rng.choice(nodes_A, size=k, replace=False))
        n0 = InterdependentNetwork(G_A, G_B, seed=42)
        n2 = InterdependentNetwork(G_A, G_B, seed=42)
        a.append(n0.run(failed)["S_A"])
        b.append(n2.run(failed, rule=L2Rule(cfg_deg))["S_A"])
    S_L0.append(np.mean(a)); S_L2.append(np.mean(b))

assert np.allclose(S_L2, S_L0, atol=1e-9), "退化性检验失败！"
print("✅ 退化性检验通过：ρ=0 且 φ=0 时 L2 ≡ L0")

## 第 3 步

In [ ]:
# 状态驱动让级联更早发生
plt.figure(figsize=(8, 5))
for label, cfg in [("ρ=0, φ=0（退回 L0）", StateDrivenConfig(rho=0.0, phi=0.0)),
                   ("ρ=0, φ=0.3（常数耦合）", StateDrivenConfig(rho=0.0, phi=0.3)),
                   ("ρ=1, φ=0.3（状态驱动）", StateDrivenConfig(rho=1.0, phi=0.3)),
                   ("ρ=2, φ=0.5（强驱动）", StateDrivenConfig(rho=2.0, phi=0.5))]:
    rng = np.random.default_rng(42)
    curve = []
    for f in fractions:
        k = int(round(len(nodes_A) * f))
        trials = []
        for _ in range(3):
            failed = ([] if k == 0 else rng.choice(nodes_A, size=k, replace=False))
            n = InterdependentNetwork(G_A, G_B, seed=42)
            trials.append(n.run(failed, rule=L2Rule(cfg))["S_A"])
        curve.append(np.mean(trials))
    plt.plot(fractions, curve, "o-", label=label)
plt.xlabel("移除比例 f"); plt.ylabel("存活序参量 S")
plt.title("L2：耦合强度随状态演化 W(t)")
plt.legend(fontsize=9); plt.grid(alpha=0.3)
plt.savefig(RESULTS / "notebook_04_L2_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

**思考题**

1. ρ 与 φ 各自代表什么现实含义？（退化惩罚 / 服务质量底线）
2. 为什么说"状态驱动耦合"比"常数耦合"更真实？举一个电网或产线的例子。
3. 如果 φ 设得比 1 还大，会发生什么？（提示：谁都达不到）